In [ ]:
import pyranges as pr
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt


In [ ]:
df_RPKM= pd.read_csv('45646_mRNA_S43_L004_R1_001_qexpr_rpkm.txt', sep='\t', header=None, 
                     names=["gene_id", "length", "mRNA reads", "RPM", "length_kb", "RPKM"])

In [ ]:
def get_rpkm(df, gene_id):
    return df.loc[df['gene_id'] == gene_id, 'RPKM'].iloc[0]

snf5_RPKM = get_rpkm(df_RPKM, "YBR289W")


In [ ]:
path = ("/mnt/unallab/jleslie/20250827_ribosome_profiling_snf5_cyc8_ade2/28_29merAnalysis/snf5analysis/45646_mono_S39_L004_R1_001_Aligned_unique_sorted_fwd_YBR289W.bed")

gr = pr.read_bed(path)
df = gr.df

In [ ]:
df

In [ ]:
region_start  = 779666
region_end    = 782381
num_positions = region_end - region_start

# Normalize first
df['Score'] = df['Name'].div(float(snf5_RPKM))
df['Score'] = df['Score'].fillna(0)

# Build score dict with all positions initialized to 0
score_dict_snf5 = {i: 0 for i in range(num_positions)}

# Fill in real scores
for index, row in df.iterrows():
    pos = row['Start'] - region_start
    if 0 <= pos < num_positions:
        score_dict_snf5[pos] = row['Score']

score_list_snf5 = list(score_dict_snf5.values())

# DataFrame for export
df_snf5 = pd.DataFrame({
    "Position": list(score_dict_snf5.keys()),
    "Score":    score_list_snf5
})

df_snf5.to_csv('041526_45646_normalized_asiteoccupancy_monosomes_snf5.csv', index=False)


print(f"Length: {len(score_list_snf5)}")
score_list_snf5


In [ ]:
before_stop = score_list_snf5[:672]
after_stop = score_list_snf5[672:]
len(after_stop) #checking length
len(before_stop) #checking length

def frame(x):
    firstframe  = x[0::3]
    secondframe = x[1::3]
    thirdframe  = x[2::3]
    total = sum(x)
    
    return {
        "first":  sum(firstframe)  / total,
        "second": sum(secondframe) / total,
        "third":  sum(thirdframe)  / total
    }

frames_before_45646 = frame(before_stop)
frames_after_45646 = frame(after_stop)
frames_before_45646

In [ ]:
scorelist_5utr = score_list_snf5[:672]
scorelist_3utr = score_list_snf5[672:]

def frame_fractions(scores):
    frame1 = scores[0::3]
    frame2 = scores[1::3]
    frame3 = scores[2::3]
    total = sum(scores)
    if total == 0:
        return {"first": 0, "second": 0, "third": 0}
    return {
        "first": sum(frame1) / total,
        "second": sum(frame2) / total,
        "third": sum(frame3) / total
    }

# Compute fractions for both regions
five_utr_frames = frame_fractions(scorelist_5utr)
three_utr_frames = frame_fractions(scorelist_3utr)

# Display comparison
print("5′ UTR Frame Fractions:", five_utr_frames)
print("3′ UTR Frame Fractions:", three_utr_frames)

In [ ]:
len(score_list), len(thirdframe)

In [ ]:
#counts for all frames
cdsstart = score_list[0]
cdsend = score_list[675]

utr5len = 0
cdslen = 675
utr3len = len(score_list) - 675
mrnalen = len(score_list)

#counts for all frames
utr3Counts = sum(score_list[cdslen:])
cdsCounts = sum(score_list[:(cdslen-1)])
mrnaCounts = cdsCounts+utr3Counts

#rpk for all frames
utr3Density_rpkm = (utr3Counts/len(score_list[cdslen:]))*1000
cdsDensity_rpkm = (cdsCounts/len(score_list[:(cdslen-1)]))*1000
mrnaDensity_rpkm = ((cdsCounts+utr3Counts)/len(score_list))*1000

#rrts for all frames
RRTS = utr3Density_rpkm/cdsDensity_rpkm
print(RRTS)